In [1]:
!pip install kafka-python pandas jaeger-client

In [2]:
import json
import pandas as pd
from kafka import KafkaConsumer
from jaeger_client import Config
from datetime import datetime
import os

In [3]:
# Initialize Jaeger tracer
config = Config(
    config={
        'sampler': {'type': 'const', 'param': 1},
        'logging': True,
    },
    service_name='consumer-classifier',
)
tracer = config.initialize_tracer()

In [4]:
# MITRE ATT&CK Classification Rules
CLASSIFICATION_RULES = {
    'port_scan': {
        'classification': 'suspicious',
        'severity': 'medium',
        'mitre_tactic': 'Discovery',
        'mitre_technique': 'T1046',
        'description': 'Network Service Scanning'
    },
    'brute_force': {
        'classification': 'malicious',
        'severity': 'high',
        'mitre_tactic': 'Credential Access',
        'mitre_technique': 'T1110',
        'description': 'Brute Force Attack'
    },
    'data_exfil': {
        'classification': 'malicious',
        'severity': 'critical',
        'mitre_tactic': 'Exfiltration',
        'mitre_technique': 'T1048',
        'description': 'Data Exfiltration'
    },
    'normal_web': {
        'classification': 'benign',
        'severity': 'none',
        'mitre_tactic': None,
        'mitre_technique': None,
        'description': 'Normal Web Traffic'
    },
    'dns_query': {
        'classification': 'benign',
        'severity': 'none',
        'mitre_tactic': None,
        'mitre_technique': None,
        'description': 'DNS Query'
    }
}

In [5]:
def classify_event(event):
    """Classify event based on rules"""
    event_type = event.get('event_type', 'unknown')
    rules = CLASSIFICATION_RULES.get(event_type, {
        'classification': 'unknown',
        'severity': 'low',
        'mitre_tactic': None,
        'mitre_technique': None,
        'description': 'Unknown Event Type'
    })
    
    classified_event = {
        **event,
        'classification': rules['classification'],
        'severity': rules['severity'],
        'mitre_tactic': rules['mitre_tactic'],
        'mitre_technique': rules['mitre_technique'],
        'description': rules['description'],
        'classified_at': datetime.utcnow().isoformat()
    }
    
    return classified_event

In [6]:
def save_to_csv(events, filename='/home/jovyan/data/classified_packets.csv'):
    """Save classified events to CSV"""
    df = pd.DataFrame(events)
    
    # Check if file exists
    if os.path.exists(filename):
        # Append to existing file
        df.to_csv(filename, mode='a', header=False, index=False)
    else:
        # Create new file
        df.to_csv(filename, index=False)
    
    return len(events)

In [7]:
def consume_and_classify(max_messages=50, timeout_ms=10000):
    """Consume messages from Kafka and classify them"""
    
    # Initialize Kafka consumer
    consumer = KafkaConsumer(
        'events.raw',
        bootstrap_servers=['kafka:9092'],
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='earliest',
        enable_auto_commit=True,
        group_id='classifier-group'
    )
    
    print(f"Consuming up to {max_messages} messages from Kafka...\n")
    
    classified_events = []
    message_count = 0
    
    try:
        for message in consumer:
            with tracer.start_span('consume_message') as span:
                event = message.value
                span.set_tag('event_id', event.get('event_id'))
                
                # Classify event
                with tracer.start_span('classify', child_of=span) as classify_span:
                    classified = classify_event(event)
                    classify_span.set_tag('classification', classified['classification'])
                    classify_span.set_tag('severity', classified['severity'])
                
                classified_events.append(classified)
                message_count += 1
                
                if message_count % 10 == 0:
                    print(f"Processed {message_count} events")
                
                # Break if we've processed enough messages
                if message_count >= max_messages:
                    break
    
    except KeyboardInterrupt:
        print("\nInterrupted by user")
    
    finally:
        consumer.close()
    
    # Save to CSV
    if classified_events:
        with tracer.start_span('save_to_storage'):
            saved_count = save_to_csv(classified_events)
            print(f"\nSaved {saved_count} classified events to CSV")
    
    print(f"\nProcessing complete!")
    print(f"Total events processed: {message_count}")
    print(f"View traces at: http://localhost:16686")
    
    return classified_events

In [8]:
# Run the consumer and classifier
classified_events = consume_and_classify(max_messages=50)

Consuming up to 50 messages from Kafka...

Processed 10 events
Processed 20 events
Processed 30 events
Processed 40 events
Processed 50 events

Saved 50 classified events to CSV

Processing complete!
Total events processed: 50
View traces at: http://localhost:16686
